# ECFFT for Group BaseFold

An interactive walkthrough of the ECFFT over the BN-254 base field, building towards the **group-valued BaseFold** protocol from [eprint 2025/1325](https://eprint.iacr.org/2025/1325).

**The problem:** In IPA verification, computing $G(r) = \sum_i s_i \cdot G_i$ is an $O(n)$ MSM that dominates recursive verification. Group BaseFold replaces this with FRI-like folding over group elements, using $O(\lambda \cdot \log^2 n)$ scalar multiplications.

**Why ECFFT?** The BN-254 base field has no roots of unity of large 2-power order, so the standard FFT doesn't apply. The ECFFT replaces the squaring map $x \mapsto x^2$ with a rational map $\psi(x) = (x-b)^2/x$ induced by a good isogeny on an elliptic curve.

**Outline:**
1. Curve infrastructure and the $\psi$ map (§1–§3)
2. FRI domain construction and the pointwise fold (§4–§5)
3. Group-valued BaseFold (§6)
4. General FFTree for ENTER/EXIT (§7–§8, in `ecfft_fftree.py`)

In [ ]:
# Core: field arithmetic, curves, isogenies, FRI domains, pointwise fold, group BaseFold
from ecfft_algorithms import (
    q, fadd, fsub, fmul, fdiv, fneg, finv, fsqrt, fpow,
    GoodCurve, Point, RationalMap,
    good_isogeny, apply_isogeny, build_isogeny_chain,
    build_fri_domains, poly_eval,
    ecfri_fold_step, ecfri_fold, ecfri_verify_query,
    basefold_group_fold_step, basefold_verify_query,
    _group_add, _group_neg, _group_scalar_mul,
)

# General FFTree (ENTER/EXIT/EXTEND/DEGREE) — not needed for BaseFold
from ecfft_fftree import (
    build_fftree, FFTree, lagrange_interpolate,
    build_evaluation_domain, split_domain_with_psi,
)

from ecfft_params_2_20 import params, verify

## 1. The curve and its parameters

We work over the BN-254 base field $\mathbb{F}_q$ with:
$$q = 21888242871839275222246405745257275088696311157297823662689037894645226208583$$

Our "good curve" has the form $E: y^2 = x^3 + a \cdot x^2 + B \cdot x$ with $B = b^2$.

The key property: $E(\mathbb{F}_q)$ contains a cyclic subgroup of order $2^{20}$.

In [ ]:
print(f"Field: F_q where q = {q}")
print(f"q mod 4 = {q % 4}  (so q ≡ 3 mod 4 — Tonelli-Shanks simplifies to x^((q+1)/4))")
print(f"")
print(f"Curve parameters:")
print(f"  a  = {params['a']}")
print(f"  B  = {params['bb']}")
print(f"  Generator order: 2^{params['k']} = {2**params['k']}")

In [ ]:
# Verify the parameters
verify()

## 2. The good isogeny and the ψ map

A "good isogeny" $\phi: E \to E'$ with kernel $\langle T \rangle$ where $T = (0, 0)$ gives us:

$$\phi(x, y) = \left(\psi(x),\ h(x) \cdot y\right)$$

where $\psi(x) = \frac{(x - b)^2}{x}$ is the x-coordinate map.

**Key property:** $\psi$ is 2-to-1 on the evaluation domain. If $P$ and $P + T$ are in the domain, then $\psi(x(P)) = \psi(x(P + T))$.

In [ ]:
curve = GoodCurve(params['a'], params['bb'])
gen = Point(params['gx'], params['gy'], curve)

# Build the first isogeny
psi, h, codomain = good_isogeny(curve)

print(f"Curve E:  y² = x³ + a·x² + B·x")
print(f"  b = √B = {curve.b}")
print(f"  ψ(x) = (x - b)² / x")
print(f"")
print(f"Codomain E':  y² = x³ + {codomain.a}·x² + {codomain.bb}·x")
print(f"  a' = a + 6b = {fadd(curve.a, fmul(6, curve.b))}")

### ψ is 2-to-1: a concrete example

Let's build a small domain (size 8) and verify that ψ maps it 2-to-1.

In [ ]:
# Scale generator to order 8 = 2^3
k = params['k']
gen8 = gen.scalar_mul(2 ** (k - 3))

# Build evaluation domain: x-coordinates of O+coset, G+coset, 2G+coset, ..., 7G+coset
coset = gen.double()
domain = build_evaluation_domain(gen8, coset, 8)
print(f"Domain (8 x-coordinates):")
for i, x in enumerate(domain):
    print(f"  s[{i}] = {x}")

In [ ]:
# Build the isogeny chain for 3 levels (log2(8) = 3)
psis, curves, hs = build_isogeny_chain(gen8, 3)

# Show ψ₀ is 2-to-1: first half pairs with second half
print("ψ₀ pairing (first half ↔ second half):")
for i in range(4):
    img_first = psis[0](domain[i])
    img_second = psis[0](domain[i + 4])
    print(f"  ψ₀(s[{i}]) = ψ₀(s[{i+4}]) = {img_first}   {'✓' if img_first == img_second else '✗'}")

## 3. The isogeny chain: recursive halving

Each ψ map halves the domain. Starting from 8 points, we get:

$$8 \xrightarrow{\psi_0} 4 \xrightarrow{\psi_1} 2 \xrightarrow{\psi_2} 1$$

This is the ECFFT analogue of the FFT butterfly: at each level, pairs of points collapse to one.

In [ ]:
current = list(domain)
for level in range(3):
    half = len(current) // 2
    s0, s1 = current[:half], current[half:]
    images = [psis[level](s0[i]) for i in range(half)]
    print(f"Level {level}: {len(current)} points → {len(images)} points via ψ_{level}")
    for i in range(half):
        assert psis[level](s0[i]) == psis[level](s1[i])
    current = images
print(f"\nFinal: {len(current)} point(s) — the recursion base case")

## 4. FRI domains and the pointwise fold

The FRI protocol operates on a sequence of evaluation domains $L_0 \to L_1 \to \cdots \to L_k$ connected by $\psi$ maps. Each $\psi_i$ is 2-to-1 on $L_i$, halving the domain.

The **ECFFT2 pointwise fold** (BSCKL22, Appendix B.2) decomposes any polynomial $P$ of degree $< d$ as:

$$P(x) = (P_0(\psi(x)) + x \cdot P_1(\psi(x))) \cdot x^{d/2 - 1}$$

For a pair $(s_0, s_1)$ with $\psi(s_0) = \psi(s_1)$, setting $e = d/2 - 1$:

$$H_z[P](t) = a + \text{slope} \cdot (z - s_0)$$

where $a = P(s_0)/s_0^e$, $b = P(s_1)/s_1^e$, $\text{slope} = (b-a)/(s_1-s_0)$.

This is **pointwise**: each output depends on exactly 2 inputs → O(1) per-query FRI verification.

In [ ]:
# Build FRI domains
layers, fri_psis = build_fri_domains(params, log_n=5)  # L_0 has 32 elements

print(f"FRI domain layers:")
for i, L in enumerate(layers):
    print(f"  L_{i}: {len(L)} elements")

# Evaluate a polynomial on L_0
coeffs32 = list(range(1, 33))
word = [poly_eval(coeffs32, x) for x in layers[0]]

# One FRI fold step: degree bound 32 → 16
z = 42
folded = ecfri_fold_step(word, layers, round_idx=0, degree_bound=32, z=z)
print(f"\nFold step: {len(word)} evals → {len(folded)} evals")

# Verify a single query (O(1)):
j = 5
expected = ecfri_verify_query(layers, 0, 32, j, word[j], word[j + 16], z)
print(f"Query verification at j={j}: {'✓' if expected == folded[j] else '✗'}")

In [ ]:
# Multi-round FRI fold
challenges = [42, 99, 7, 13, 256]
fully_folded = ecfri_fold(word, layers, degree_bound=32, challenges=challenges)
print(f"Multi-round fold: {len(word)} evals → {len(fully_folded)} eval(s) after {len(challenges)} rounds")

# Verify all queries across all rounds
all_ok = True
for j_query in range(len(layers[-1])):
    current_word = list(word)
    d = 32
    for rnd, z_r in enumerate(challenges):
        m = len(layers[rnd])
        half = m // 2
        exp_val = ecfri_verify_query(layers, rnd, d, j_query % half,
                                      current_word[j_query % half],
                                      current_word[(j_query % half) + half], z_r)
        current_word = ecfri_fold_step(current_word, layers, rnd, d, z_r)
        if exp_val != current_word[j_query % (half // 2 if half > 1 else 1)]:
            all_ok = False
        d //= 2
print(f"All query verifications: {'✓' if all_ok else '✗'}")

## 5. Group-valued BaseFold

The same pointwise fold formula works over **group elements** instead of field elements. "Division" by a scalar becomes multiplication by its inverse; "addition" is elliptic curve addition.

This is the core of the group BaseFold protocol: fold a vector of group elements (the SRS encoding) through $k = \log n$ rounds, then verify by spot-checking fold consistency at random positions.

Each spot-check costs 4 scalar multiplications per round — independent of $n$.

In [ ]:
# For this demo, we use a simple "group" of (x, y) points.
# In practice these would be Grumpkin points; here we just fabricate
# group elements by scalar-multiplying a base point.

# Use a small domain for speed
layers_small, _ = build_fri_domains(params, log_n=3)  # L_0 has 8 elements
n_small = len(layers_small[0])

# Fabricate group elements: G_i = i * base_point  (using a made-up base)
# We use a point on y² = x³ + 3 (a = 0 Weierstrass, like Grumpkin)
# base_point = (1, sqrt(1 + 3)) = (1, 2) over Fq
base_x = 1
base_y = fsqrt(fadd(fmul(base_x, fmul(base_x, base_x)), 3))
assert base_y is not None, "Need a valid curve point"
base_pt = (base_x, base_y)

# "SRS encoding": g_word[j] = sum_i L_0[j]^i * G_i  (simplified: just use scalar * base)
g_word = []
for j in range(n_small):
    # Use a simple scalar derived from the domain point
    scalar = (layers_small[0][j] * (j + 1)) % q
    g_word.append(_group_scalar_mul(base_pt, scalar))

print(f"Group-element word: {n_small} points on L_0")

# Fold one round
z_group = 42
g_folded = basefold_group_fold_step(g_word, layers_small, round_idx=0, degree_bound=8, z=z_group)
print(f"After fold: {len(g_folded)} points on L_1")

# Verify a query
j = 2
expected_g = basefold_verify_query(layers_small, 0, 8, j, g_word[j], g_word[j + 4], z_group)
match = expected_g == g_folded[j]
print(f"Query verification at j={j}: {'✓' if match else '✗'}")
print(f"  expected: {expected_g}")
print(f"  got:      {g_folded[j]}")

---

## Appendix A: General FFTree (ENTER / EXIT / EXTEND / DEGREE)

The FFTree provides $O(n \log^2 n)$ polynomial evaluation and interpolation — the general-purpose ECFFT engine from Part I. This is **not needed for group BaseFold** but is useful for polynomial commitment schemes.

- **ENTER**: coefficients → evaluations (analogue of FFT)
- **EXIT**: evaluations → coefficients (analogue of inverse FFT)
- **DEGREE**: degree from evaluations in $O(n \log n)$
- **EXTEND**: evaluations on one moiety → the other moiety

In [ ]:
# Build an FFTree of size 8
tree, leaves = build_fftree(params, log_n=3)
eval_domain = tree.eval_domain()

# A polynomial: f(x) = 1 + 2x + 3x² + ... + 8x⁷
coeffs = list(range(1, 9))
print(f"f(x) = {' + '.join(f'{c}x^{i}' if i > 0 else str(c) for i, c in enumerate(coeffs))}")

# ENTER: fast evaluation
evals = tree.enter(coeffs)
naive = [poly_eval(coeffs, x) for x in eval_domain]
print(f"ENTER: {'✓' if evals == naive else '✗'}  (matches naive)")

# EXIT: fast interpolation (inverse of ENTER)
recovered = tree.exit(evals)
print(f"EXIT:  {'✓' if recovered == coeffs else '✗'}  (round-trip)")

# DEGREE
deg = tree.degree(evals)
print(f"DEGREE: {deg}  (expected 7)")

# Lower-degree polynomial
low_coeffs = [1, 2, 3] + [0] * 5
low_evals = tree.enter(low_coeffs)
print(f"DEGREE of 1+2x+3x²: {tree.degree(low_evals)}  (expected 2)")

In [ ]:
# EXTEND: given evaluations on S₀ (even-indexed domain points),
# compute evaluations on S₁ (odd-indexed points).

tree16, _ = build_fftree(params, log_n=4)  # size 16
dom16 = tree16.eval_domain()
s0_points = dom16[0::2]
s1_points = dom16[1::2]

poly7 = list(range(1, 9))  # degree 7 (< 16/2 = 8)
s0_vals = [poly_eval(poly7, x) for x in s0_points]

s1_extended = tree16.extend(s0_vals, 'S1')
s1_naive = [poly_eval(poly7, x) for x in s1_points]

print(f"EXTEND S₀ → S₁: {'✓' if s1_extended == s1_naive else '✗'}")

### Scaling up

In [ ]:
import time

for log_n in [3, 4, 5, 6]:
    n = 1 << log_n
    tree_n, _ = build_fftree(params, log_n)
    dom_n = tree_n.eval_domain()
    coeffs_n = list(range(1, n + 1))

    t0 = time.time()
    evals_n = tree_n.enter(coeffs_n)
    t_enter = time.time() - t0

    t0 = time.time()
    recovered_n = tree_n.exit(evals_n)
    t_exit = time.time() - t0

    ok = recovered_n == coeffs_n
    naive_n = [poly_eval(coeffs_n, x) for x in dom_n]
    enter_ok = evals_n == naive_n

    print(f"n={n:4d}  ENTER: {t_enter:.3f}s {'✓' if enter_ok else '✗'}  "
          f"EXIT: {t_exit:.3f}s {'✓' if ok else '✗'}")

### Multiple curve parameter sets

We have parameters for curves with cyclic subgroups of order $2^{18}$, $2^{19}$, and $2^{20}$.

In [ ]:
from ecfft_params_2_18 import params as p18
from ecfft_params_2_19 import params as p19
from ecfft_params_2_20 import params as p20

for name, p in [("2^18", p18), ("2^19", p19), ("2^20", p20)]:
    tree_p, _ = build_fftree(p, log_n=4)
    dom_p = tree_p.eval_domain()
    c = list(range(1, 17))
    e = tree_p.enter(c)
    r = tree_p.exit(e)
    naive_p = [poly_eval(c, x) for x in dom_p]
    print(f"Curve {name}:  ENTER {'✓' if e == naive_p else '✗'}  EXIT {'✓' if r == c else '✗'}  "
          f"DEGREE {tree_p.degree(e)}")